# **Model**

# **XRF milk database**

In [73]:
# importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks

# loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/milk/plsda/milk.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '2.66':'22.62']

# Creating a new column 'Class' based on the condition of the samples in the 'Type' column being 'Authentic'
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.66':'22.62'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.66':'22.62'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=4,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-27 08:15:16,184 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-27 08:15:16,206 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

# **VIP and SHAP**

In [74]:
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 
Xcalclass.T.plot()

In [ ]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Ag La', 2.66, 3.10),
('Ag Lb', 3.10, 3.46),
('Ca', 3.46, 3.92),
('background', 3.92, 6.12),
('Fe', 6.12, 6.68),
('Cu', 6.70, 8.37),
('Zn', 8.37, 9.10),
('Bremsstrahlung', 9.10, 20.06),
('Ag compton', 20.06, 21.62),
('Ag ka', 21.48, 22.62)
]

In [76]:
# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [63]:
# calculando a covariancia entre cada variável espectral e a predição do modelo PLS-DA
cov_scores = []
y_pred = plsda_results[5].iloc[:,-1].values # using the continuous predictions from LV=3
for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [77]:
import numpy as np
import pandas as pd

# vip
vip_scores_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'VIP_Score' : plsda_results[4].T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_vip = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in vip_scores_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_vip[i] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# reg vet
reg_vet = pd.DataFrame(plsda_results[3].coef_, columns=plsda_results[3].feature_names_in_) # creating a DataFrame with regression coefficients
reg_vet = reg_vet.T
reg_vet.insert(0, 'energy', reg_vet.index) # adding energy column
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy', 'Reg_coef'] # renaming
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs() # adding absolute value column
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True) # sorting by absolute value

# gerando uma nova coluna em reg_vet com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_reg = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
    for i in reg_vet['energy']: # iterando sobre cada valor de energia no reg_vet
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_reg[i] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_reg
reg_vet

# vamos filtrar reg_vet para manter apenas as zonas espectrais únicas com maior valor absoluto do coeficiente de regressão
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)  

# **bagging - covariance**

In [78]:
import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ag La <= 4.26,Ag La <= 4.26,Ag La <= 4.26,Ag La <= 4.26
1,Ag La <= 2.83,Ag La <= 2.83,Ag La <= 2.83,Ag La <= 2.83
2,Ag La <= 5.23,Ag La <= 5.23,Ag La <= 5.23,Ag La <= 5.23
3,Ag Lb <= 3.70,Ag Lb <= 3.70,Ag Lb <= 3.70,Ag Lb <= 3.70
4,Ag Lb <= 4.33,Ag Lb <= 4.33,Ag Lb <= 4.33,Ag Lb <= 2.77
...,...,...,...,...
56,Fe > 1.63,Zn <= -1.35,Class_A,background1 > 2.38
57,Class_A,background1 > 2.38,Class_B,Class_A
58,Class_B,Cu > 2.01,NaN,Class_B
59,NaN,Class_A,NaN,NaN


In [79]:
all_results_cov[0]['cov_results_dict']['Bag_1']

,Predicate,Covariance
0,Ag La <= 4.26,1.498562
1,Ag La <= 2.83,1.436467
2,Ag La <= 5.23,1.231677
3,Ag Lb <= 3.70,1.216625
4,Ag Lb <= 2.77,1.060606
5,Ag Lb <= 4.33,0.999557
6,Ag ka <= 10.15,0.667310
7,Ag ka <= 5.20,0.422118
8,Ag ka <= -3.39,0.394511
9,Ag ka > -3.39,0.387425


In [80]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 4.26,16.538775,Ag La,4.26,<=
1,Ag Lb <= 3.70,8.385409,Ag Lb,3.70,<=
2,Ag ka <= 10.15,5.319056,Ag ka,10.15,<=
3,Zn > -1.89,0.651477,Zn,-1.89,>
4,Ag compton <= 3.09,0.603257,Ag compton,3.09,<=
5,Fe <= 2.10,0.582293,Fe,2.10,<=
6,Ca <= 1.99,0.443690,Ca,1.99,<=
7,Bremsstrahlung1 <= 3.06,0.411927,Bremsstrahlung1,3.06,<=
8,Cu <= 2.01,0.325747,Cu,2.01,<=
9,background1 <= 2.38,0.319127,background1,2.38,<=


In [81]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zone_sums_df_original = exp.aggregate_spectral_zones(spectral_zones_original, aggregator='extreme')

# Aplicar o mapeamento
lrc_summed_unique_df_cov_with_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_unique_df_cov,
    zone_sums_preprocessed=zone_sums_df,
    zone_sums_natural=zone_sums_df_original
)

lrc_summed_unique_df_cov_with_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 4.26,16.538775,Ag La,4.26,<=,310.0000,38.0,0.004828,Ag La <= 310.000000
1,Ag Lb <= 3.70,8.385409,Ag Lb,3.70,<=,205.8950,142.0,0.000239,Ag Lb <= 205.895000
2,Ag ka <= 10.15,5.319056,Ag ka,10.15,<=,3290.4200,57.0,0.035126,Ag ka <= 3290.420000
3,Zn > -1.89,0.651477,Zn,-1.89,>,42.7895,7.0,0.001592,Zn > 42.789500
4,Ag compton <= 3.09,0.603257,Ag compton,3.09,<=,707.7890,40.0,0.000539,Ag compton <= 707.789000
5,Fe <= 2.10,0.582293,Fe,2.10,<=,21.7895,19.0,0.000690,Fe <= 21.789500
6,Ca <= 1.99,0.443690,Ca,1.99,<=,36.3684,117.0,0.003036,Ca <= 36.368400
7,Bremsstrahlung1 <= 3.06,0.411927,Bremsstrahlung1,3.06,<=,675.0000,164.0,0.001614,Bremsstrahlung1 <= 675.000000
8,Cu <= 2.01,0.325747,Cu,2.01,<=,36.5263,103.0,0.001116,Cu <= 36.526300
9,background1 <= 2.38,0.319127,background1,2.38,<=,15.0000,73.0,0.004027,background1 <= 15.000000


# **Perturbation**

In [90]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='mean', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
PERTURBATION IMPORTANCE PARA PREDICADOS
Modo de perturbação: mean
Fonte das estatísticas: full
Métrica: mean_abs_diff
Total de folds: 10


[Bag_1] Processando 60 predicados..

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ag La <= 2.83,Ag La <= 2.83,Ag La <= 2.83,Ag La <= 2.83
1,Ag La <= 4.26,Ag La <= 4.26,Ag La <= 4.26,Ag La <= 4.26
2,Ag La <= 5.23,Ag La <= 5.23,Ag La <= 5.23,Ag La <= 5.23
3,Ag La > 4.26,Ag La > 4.26,Ag La > 4.26,Ag La > 4.26
4,Ag ka <= 5.20,Ag ka > 5.20,Ag ka <= -3.39,Ag ka <= 5.20
...,...,...,...,...
57,Cu > -2.02,Ca <= 1.99,background1 > 2.11,Cu > 2.35
58,Ca <= 2.47,Ca <= 2.47,Cu > -2.02,Cu <= 2.71
59,Cu <= 2.01,Cu <= 2.01,Ca <= -1.26,background1 <= 2.73
60,Class_A,Class_A,Class_A,Class_A


In [83]:
all_results_pert[0]['pert_results_dict']['Bag_5']

,Predicate,Perturbation
0,Ag La <= 2.83,0.219963
1,Ag La <= 4.26,0.181044
2,Ag La <= 5.23,0.168101
3,Ag ka <= -3.39,0.152528
4,Ag La > 4.26,0.144985
5,Ag ka > 5.20,0.135153
6,Ag La > 2.83,0.130386
7,Ag Lb <= 2.77,0.128570
8,Ag La > -8.20,0.126992
9,Ag ka <= 5.20,0.126719


In [91]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 2.83,4.415740,Ag La,2.83,<=
1,Ag ka <= -3.39,2.125566,Ag ka,-3.39,<=
2,Ag Lb <= 2.77,2.112727,Ag Lb,2.77,<=
3,Bremsstrahlung1 > -3.10,1.833634,Bremsstrahlung1,-3.10,>
4,Ag compton <= -2.03,0.991817,Ag compton,-2.03,<=
5,Zn > 1.85,0.353998,Zn,1.85,>
6,Fe > 1.63,0.208031,Fe,1.63,>
7,background1 <= 2.38,0.060124,background1,2.38,<=
8,Ca <= 1.99,0.053569,Ca,1.99,<=
9,Cu > 2.35,0.051271,Cu,2.35,>


In [92]:
lrc_summed_df

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 2.83,4.415740,Ag La,2.83,<=
1,Ag La <= 4.26,3.665127,Ag La,4.26,<=
2,Ag La <= 5.23,3.006098,Ag La,5.23,<=
3,Ag La > 4.26,2.439751,Ag La,4.26,>
4,Ag ka <= -3.39,2.125566,Ag ka,-3.39,<=
...,...,...,...,...,...
57,Cu > -2.02,0.044722,Cu,-2.02,>
58,Cu <= 2.01,0.032486,Cu,2.01,<=
59,Ca <= 2.47,0.029383,Ca,2.47,<=
60,Class_B,0.000000,None,None,None


In [93]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zone_sums_df_original = exp.aggregate_spectral_zones(spectral_zones_original, aggregator='extreme')

# Aplicar o mapeamento
lrc_summed_unique_df_pert_with_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_unique_df_pert,
    zone_sums_preprocessed=zone_sums_df,
    zone_sums_natural=zone_sums_df_original
)

lrc_summed_unique_df_pert_with_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 2.83,4.415740,Ag La,2.83,<=,255.0000,58.0,0.002861,Ag La <= 255.000000
1,Ag ka <= -3.39,2.125566,Ag ka,-3.39,<=,3452.0000,29.0,0.008152,Ag ka <= 3452.000000
2,Ag Lb <= 2.77,2.112727,Ag Lb,2.77,<=,179.0000,95.0,0.003992,Ag Lb <= 179.000000
3,Bremsstrahlung1 > -3.10,1.833634,Bremsstrahlung1,-3.10,>,634.0000,245.0,0.000855,Bremsstrahlung1 > 634.000000
4,Ag compton <= -2.03,0.991817,Ag compton,-2.03,<=,641.4210,251.0,0.001921,Ag compton <= 641.421000
5,Zn > 1.85,0.353998,Zn,1.85,>,44.2632,6.0,0.000309,Zn > 44.263200
6,Fe > 1.63,0.208031,Fe,1.63,>,19.4737,250.0,0.002499,Fe > 19.473700
7,background1 <= 2.38,0.060124,background1,2.38,<=,15.0000,73.0,0.004027,background1 <= 15.000000
8,Ca <= 1.99,0.053569,Ca,1.99,<=,36.3684,117.0,0.003036,Ca <= 36.368400
9,Cu > 2.35,0.051271,Cu,2.35,>,38.7895,220.0,0.000164,Cu > 38.789500


In [94]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    #len(shap_unique_df['Zone']),
    len(lrc_summed_unique_df_pert_with_natural['Zone']),
    len(lrc_summed_unique_df_cov_with_natural['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Vip': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_coef': pad_list(reg_vet_unique_df['Zone'], max_len),
    #'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert_with_natural['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov_with_natural['Zone'], max_len),
})

#features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Vip,Reg_coef,LRC_perturbation,LRC_covariance
0,Ag La,Ag ka,Ag La,Ag La
1,Ag Lb,Zn,Ag ka,Ag Lb
2,Ag ka,Ag La,Ag Lb,Ag ka
3,Zn,Fe,Bremsstrahlung1,Zn
4,Ag compton,Ag Lb,Ag compton,Ag compton
5,Fe,Bremsstrahlung1,Zn,Fe
6,Bremsstrahlung1,Ag compton,Fe,Ca
7,Ca,Ca,background1,Bremsstrahlung1
8,background1,background1,Ca,Cu
9,Cu,Cu,Cu,background1


In [95]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if x is not None]
methods = ['Reg_coef', 'LRC_covariance', 'LRC_perturbation']
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if x is not None]
    # Truncate both lists to the same length (minimum of both)
    min_len = min(len(reference_list), len(compare_list))
    ref_trunc = reference_list[:min_len]
    cmp_trunc = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trunc, cmp_trunc).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
#rbo_results.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_results

,Reference,Method,RBO_Score
1,Vip,LRC_covariance,0.964789
2,Vip,LRC_perturbation,0.815130
0,Vip,Reg_coef,0.364218
